In [ ]:
# Mount Google Drive to access input/output data
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
# Install required geospatial libraries
# rioxarray : raster I/O and CRS-aware clipping built on xarray + rasterio
# cartopy   : coordinate reference system handling (dependency)
!pip install rioxarray cartopy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.2/62.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 82.9 MB/s eta 0:00:00


In [ ]:
# ============================================================
# Import Required Libraries
# ============================================================
# xarray    - labeled N-dimensional array operations
# rioxarray - geospatial extension for xarray (CRS, clipping, raster I/O)
# numpy     - numerical array operations
# ============================================================
import xarray as xr
import rioxarray as rxr
import numpy as np


In [ ]:
# ============================================================
# Batch Land Masking of ECOSTRESS LST GeoTIFFs
# ============================================================
# Applies a high-resolution land mask to a folder of ECOSTRESS
# LST GeoTIFFs, retaining only ocean/water pixels and saving
# each result as a new GeoTIFF prefixed with 'masked_'.
#
# Land mask source:
#   GSHHG (Global Self-consistent, Hierarchical, High-resolution
#   Geography) full-resolution Level 1 land polygons (GSHHS_f_L1).
#   Reference: Wessel & Smith (1996), J. Geophys. Res.
#   Download: https://www.soest.hawaii.edu/pwessel/gshhg/
#
# Masking approach:
#   rioxarray.rio.clip() with invert=True clips the raster to
#   the complement of the land polygons, keeping only water pixels.
#
# CRS handling:
#   The land shapefile is reprojected on first use to match the
#   CRS of the input rasters, then reused for all subsequent files.
# ============================================================

import rioxarray as rxr
import geopandas as gpd
import os

# ===================================================================
# Part 0: Setup Input and Output Folders
# ===================================================================

# Set input_folder to the directory containing your renamed GeoTIFFs.
input_folder  = "/content/drive/MyDrive/<YOUR_PROJECT_FOLDER>/ECOSTRESS Tifs/Geolocated LST"

# Masked outputs are written to a 'landmasked_output' subfolder.
output_folder = os.path.join(input_folder, "landmasked_output")
os.makedirs(output_folder, exist_ok=True)

# ===================================================================
# Part 1: Download and Load the GSHHG Land Mask (Runs Once)
# ===================================================================

# GSHHG shapefile archive — downloaded once and cached in the Colab runtime.
# If the zip already exists (e.g., from a previous run), the download is skipped.
zip_file = "gshhg-shp-2.3.7.zip"
if not os.path.exists(zip_file):
    os.system(f"curl -L https://www.soest.hawaii.edu/pwessel/gshhg/{zip_file} -o {zip_file}")

# Unzip only if the output directory doesn't already exist
shapefile_dir = "GSHHS_shp"
if not os.path.exists(shapefile_dir):
    os.system(f"unzip -o {zip_file}")

# Load the full-resolution (f) Level 1 (L1 = land) polygon layer.
# Level 1 represents land boundaries; full resolution is used to
# preserve accuracy along complex coastlines.
land_shapefile_path = os.path.join(shapefile_dir, "f/GSHHS_f_L1.shp")
land_gdf = gpd.read_file(land_shapefile_path)

# ===================================================================
# Part 2: Batch Process — Mask Each GeoTIFF and Save
# ===================================================================

tif_files = [f for f in os.listdir(input_folder) if f.lower().endswith('.tif')]

for tif_file in tif_files:
    input_path  = os.path.join(input_folder,  tif_file)
    output_path = os.path.join(output_folder, f"masked_{tif_file}")

    # Open the GeoTIFF as a masked xarray DataArray.
    # masked=True converts the raster's nodata value to NaN.
    # squeeze() removes the redundant band dimension (ECOSTRESS L2T is single-band).
    ecostress_data = rxr.open_rasterio(input_path, masked=True).squeeze()

    # Reproject the land mask to match the raster CRS if needed.
    # This is done in-place on the first file where a mismatch is detected;
    # subsequent files reuse the already-reprojected GeoDataFrame.
    if land_gdf.crs != ecostress_data.rio.crs:
        land_gdf = land_gdf.to_crs(ecostress_data.rio.crs)

    # Clip to the inverse of land polygons — keeps only water/ocean pixels.
    # Pixels outside the water area are set to NaN (the raster's nodata value).
    ecostress_masked = ecostress_data.rio.clip(land_gdf.geometry, invert=True)

    # Save the masked raster. CRS and spatial metadata are carried over
    # automatically from the source DataArray by rioxarray.
    ecostress_masked.rio.to_raster(output_path)


--- Part 0: Setting up folders ---
Input folder: /content/drive/MyDrive/Coral PDRDF/ECOSTRESS Tifs/Geolocated LST
Output will be saved to: /content/drive/MyDrive/Coral PDRDF/ECOSTRESS Tifs/Geolocated LST/landmasked_output

--- Part 1: Setting up the land mask ---
GSHHG zip file already exists.
Shapefile directory already exists.
Successfully loaded land shapefile.

--- Part 2: Starting batch processing ---
Found 48 .tif files to process.

Processing: ECO_L2T_LSTE.002_LST_doy2023294085443_aid0001_16N_20231021_0254.tif...
Reprojecting land mask to match ECO_L2T_LSTE.002_LST_doy2023294085443_aid0001_16N_20231021_0254.tif's CRS...
Applying land mask...
Saving masked file to: /content/drive/MyDrive/Coral PDRDF/ECOSTRESS Tifs/Geolocated LST/landmasked_output/masked_ECO_L2T_LSTE.002_LST_doy2023294085443_aid0001_16N_20231021_0254.tif
...Done.

Processing: ECO_L2T_LSTE.002_LST_doy2024074231352_aid0001_16N_20240314_1713.tif...
Applying land mask...
Saving masked file to: /content/drive/MyDrive/C